# B2.6 · Failure taxonomy

**Function B — Application Security with an AI SDLC → The Harness that Runs the SDLC**  ·  *AI for Security*

Builds on **[B2.5 · Sub-agents and delegation depth](https://spbreed.github.io/cyber-commons/lessons/B2.5.html)**.

| | |
|---|---|
| Open-source tooling | OpenTelemetry |
| Open-weight models | — |
| Frontier models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off — and where a lesson involves a model, the same code calls an open-weight endpoint or a frontier API when you configure one.

## 1 · The hook

Harnesses fail in a small number of recognisable ways, and every one of them has a signature you can detect. Without the catalogue, each failure looks like a new and mysterious problem with the model.

> **At CyberTravels.** When the review harness reports a fix that did not fix anything, the useful question is which of six named failure modes it was — not whether the model is any good.

## 2 · The framework

```
   failure                signature you can detect
   +--------------------+ +----------------------------------+
   | loop               | | same action repeated, no progress |
   | drift              | | later steps stop citing the goal  |
   | hallucinated succes| | claims done, verifier never ran   |
   | silent truncation  | | context at limit, no error raised |
   +--------------------+ +----------------------------------+

   without the catalogue, every failure looks like a new mystery
```

"The agent messed up" is not a defect report. It routes to nobody, and every
incident feels novel.

A failure taxonomy fixes that by making the class determine the **owner** and
the **fix**. Seven classes cover almost everything an agentic system does wrong:

| Class | What happened | Who fixes it |
|---|---|---|
| capability | the model could not do it | better model or better context |
| **verification** | it did it wrong and we believed it | harness engineer (B2.2) |
| authority | it did something it should not be able to do | identity (A2) |
| containment | the action reached further than intended | platform (A3) |
| injection | it was told to by untrusted content | provenance (A2.6) |
| budget | it never stopped | harness engineer (A3.4) |
| idempotency | it did the right thing twice | harness engineer (B2.8) |

The most important distinction in the table is between **capability** and
**verification**, because they look identical from the outside and have
completely different fixes. A capability failure means the model produced
something wrong. A verification failure means *your harness shipped it*. The
second is your defect, not the model's.

## 3 · Demo — classify eight real incidents

In [ ]:
INCIDENTS = [
 ("agent's patch did not compile; the loop retried and fixed it", None),
 ("agent's patch passed CI and introduced a SQL injection", None),
 ("agent deleted a production table it should never have had access to", None),
 ("agent posted the contents of .env to a public issue", None),
 ("agent approved a PR because a code comment told it to", None),
 ("agent looped for 6 hours re-running the same failing test", None),
 ("agent opened the same pull request 14 times", None),
 ("agent could not solve the task and correctly reported failure", None),
]
TAXONOMY = {
 "capability":   ("the model could not do it",              "better model / better context"),
 "verification": ("it did it wrong and we believed it",     "harness engineer — B2.2"),
 "authority":    ("it did what it should not be able to do","identity — A2"),
 "containment":  ("the action reached further than intended","platform — A3"),
 "injection":    ("untrusted content drove it",             "provenance — A2.6"),
 "budget":       ("it never stopped",                       "harness engineer — A3.4"),
 "idempotency":  ("it did the right thing twice",           "harness engineer — B2.8"),
}
LABELS = ["capability", "verification", "authority", "containment",
          "injection", "budget", "idempotency", "capability"]

for (text, _), label in zip(INCIDENTS, LABELS):
    what, owner = TAXONOMY[label]
    print(f"{label:13s} {text}")
    print(f"{'':13s} → {owner}")

## 4 · Where it breaks — the two that get confused

Incidents 1 and 2 both start "the agent's patch was wrong". They are different defects with different owners, and conflating them is how a team spends a quarter upgrading models to fix a verifier.

In [ ]:
def classify(produced_wrong_output, harness_accepted_it, action_taken):
    """The decision rule that separates capability from verification."""
    if not produced_wrong_output:
        return "not a model failure"
    if not harness_accepted_it:
        return "capability — the harness caught it, the loop worked"
    if action_taken:
        return "VERIFICATION — the harness shipped wrong work"
    return "verification (contained) — accepted but nothing acted on it"

CASES = [
 ("patch did not compile, loop retried",  True,  False, False),
 ("patch passed CI, shipped SQLi",        True,  True,  True),
 ("patch wrong, accepted, never merged",  True,  True,  False),
 ("patch correct",                        False, True,  True),
]
for name, wrong, accepted, acted in CASES:
    print(f"{name:38s} → {classify(wrong, accepted, acted)}")
print("\nThe model was equally wrong in the first three. Only one is YOUR defect.")

## 5 · The control — classify automatically from the trace

The taxonomy is only useful if applying it is cheap. Most of the classification is derivable from what the harness already records.

In [ ]:
def classify_from_trace(trace):
    """trace: dict of facts the harness already has."""
    if trace.get("denied_by_policy"):        return "authority"
    if trace.get("denied_by_sandbox"):       return "containment"
    if trace.get("instruction_source") not in (None, "principal"):
        return "injection"
    if trace.get("stopped_by", "").startswith(("step budget", "time budget")):
        return "budget"
    if trace.get("duplicate_effect"):        return "idempotency"
    if trace.get("verifier_passed") and trace.get("outcome_wrong"):
        return "verification"
    if trace.get("outcome_wrong"):           return "capability"
    return "success"

TRACES = [
 {"verifier_passed": False, "outcome_wrong": True, "stopped_by": "step budget (5 steps)"},
 {"verifier_passed": True,  "outcome_wrong": True},
 {"denied_by_policy": True},
 {"denied_by_sandbox": True},
 {"instruction_source": "pull-request-diff"},
 {"stopped_by": "time budget (300s)"},
 {"duplicate_effect": True},
 {"verifier_passed": True,  "outcome_wrong": False},
]
for t in TRACES:
    cls = classify_from_trace(t)
    owner = TAXONOMY.get(cls, ("", "—"))[1]
    print(f"{cls:14s} {owner:32s} {t}")

counts = {}
for t in TRACES:
    c = classify_from_trace(t); counts[c] = counts.get(c, 0) + 1
print(f"\ndistribution: {counts}")
assert classify_from_trace(TRACES[1]) == "verification"

## What you just proved

The eight incidents classify across all seven classes with a named owner each. The capability-vs-verification rule separates the compile failure (capability — the loop worked) from the shipped SQL injection (verification — your defect). Automatic classification from trace facts reproduces the same labels.

## Your turn

Take your last five agent incidents and assign exactly one class to each. Any incident that seems to need two classes is really two incidents, and separating them usually reveals that one of them was never fixed.

---

**Next → [B2.7 · Self-improving scaffolds](https://spbreed.github.io/cyber-commons/lessons/B2.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/B2.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/B2.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*